<a href="https://colab.research.google.com/github/ADCO02/trabajoAprendizajeProfundo/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo Aprendizaje Profundo

Ruben García y Adrian de Celis

## Objetivo del trabajo

El objetivo de este trabajo es estudiar cómo pueden aplicarse las técnicas de aprendizaje profundo a la generación de voz, centrándonos en los sistemas modernos de text-to-speech y, en particular, en los modelos capaces de realizar síntesis multilingüe y clonación de voz. Para ello se utilizará la biblioteca Coqui TTS como marco práctico y se analizará el modelo XTTS como caso de estudio representativo.

A lo largo del trabajo se explicará, primero, el fundamento teórico de la generación de voz mediante redes neuronales profundas, describiendo las principales etapas que intervienen en un sistema TTS moderno. Después, se estudiará cómo estas ideas se materializan en una librería real de uso práctico, mostrando qué tipos de modelos incluye Coqui TTS y qué capacidades ofrece para generar voz natural a partir de texto.

Además de la parte teórica, el trabajo incorporará una implementación en Python orientada a la inferencia con modelos preentrenados, así como una serie de experimentos para analizar la calidad de la voz generada, la capacidad de clonación de voz a partir de un audio de referencia y el comportamiento del sistema en distintos idiomas. Finalmente, se discutirán las limitaciones técnicas, los posibles sesgos y las implicaciones éticas derivadas del uso de este tipo de tecnologías.

En conjunto, el propósito del trabajo no es solo mostrar que un modelo puede convertir texto en audio, sino entender qué componentes hacen posible esa generación, qué grado de control ofrecen los modelos actuales sobre la identidad y el estilo de la voz y qué retos técnicos y sociales plantea esta tecnología.

## TTS y voice generation

El término TTS (Text-to-Speech) hace referencia a los sistemas capaces de transformar texto escrito en una señal de voz sintética. El objetivo de estos sistemas es producir un audio inteligible y natural, respetando tanto el contenido lingüístico del texto como propiedades prosódicas como el ritmo, la entonación, las pausas o el énfasis.

Tradicionalmente, la síntesis de voz se abordaba mediante sistemas basados en reglas o mediante síntesis concatenativa, en los que se unían fragmentos de voz previamente grabados. Sin embargo, estos enfoques tenían limitaciones importantes en naturalidad, flexibilidad y capacidad de generalización. La aparición del aprendizaje profundo supuso un cambio significativo, ya que permitió entrenar modelos capaces de aprender directamente la relación entre texto y habla a partir de grandes cantidades de datos.

Dentro de este contexto, el concepto de voice generation puede entenderse como un término más amplio que engloba distintas tareas relacionadas con la generación artificial de voz. Entre ellas se encuentran:

* la síntesis de voz propiamente dicha, donde el sistema genera audio a partir de texto;
* la clonación de voz, donde además se intenta que la voz generada conserve las características de un hablante concreto a partir de una muestra de referencia;
* y la conversión de voz, donde una grabación de un hablante se transforma para que suene como la de otro.

Por tanto, mientras que TTS describe de forma específica la tarea de convertir texto en habla, voice generation abarca un conjunto más amplio de técnicas destinadas a crear, modificar o controlar la voz sintética. En los modelos más recientes, estas tareas ya no están completamente separadas, sino que se combinan en sistemas que permiten controlar simultáneamente el contenido del mensaje, el idioma y la identidad vocal del hablante.

En este trabajo se pondrá especial atención en los modelos que no solo generan voz a partir de texto, sino que además permiten condicionar la salida con una voz de referencia, lo que abre la puerta a aplicaciones como la personalización de asistentes virtuales, la localización de contenidos en múltiples idiomas o la accesibilidad para usuarios con necesidades específicas.

## Arquitectura TTS

Aunque los sistemas de síntesis de voz han evolucionado mucho en los últimos años, la mayoría de arquitecturas modernas comparten una estructura general compuesta por varias etapas. De forma simplificada, un sistema TTS recibe una secuencia de texto como entrada y produce una señal de audio como salida, pero entre ambos extremos suelen existir representaciones intermedias que facilitan el aprendizaje del proceso.

La primera etapa consiste en el procesamiento del texto. Aquí el texto original se normaliza y transforma en una representación adecuada para el modelo. Esta fase puede incluir tareas como expandir abreviaturas, convertir números a palabras, gestionar signos de puntuación o transformar caracteres en unidades fonéticas o subpalabras. El objetivo es ofrecer al sistema una representación lingüística que capture con claridad qué debe pronunciarse.

A continuación, muchos sistemas generan una representación acústica intermedia, normalmente un mel-spectrograma. Esta representación resume cómo evoluciona la energía de la señal en distintas bandas de frecuencia a lo largo del tiempo y resulta más sencilla de modelar que la forma de onda cruda. En otras palabras, el modelo aprende primero a “imaginar” la estructura acústica de la frase antes de convertirla en sonido real.

La siguiente etapa es el vocoder, que se encarga de transformar esa representación acústica en una señal de audio final. El vocoder tiene un papel crucial en la calidad perceptiva de la voz generada, ya que de él dependen aspectos como la naturalidad, la nitidez y la ausencia de artefactos. En los sistemas modernos, tanto el predictor acústico como el vocoder suelen estar implementados mediante redes neuronales profundas especializadas.

En modelos más avanzados, especialmente los de tipo multi-speaker o multilingual, se añaden también mecanismos de condicionamiento. Esto significa que, además del texto, el modelo recibe información adicional, como por ejemplo:

* la identidad del hablante,
* el idioma en que debe sintetizar,
* el estilo de pronunciación,
* o un embedding extraído de un audio de referencia.

Este condicionamiento permite controlar características de la voz generada sin modificar el contenido textual. Así, dos entradas con el mismo texto pueden dar lugar a audios muy distintos si cambia el hablante o el idioma especificado.

Desde el punto de vista conceptual, un sistema TTS moderno puede resumirse así:

texto → representación lingüística → representación acústica → vocoder → audio

Cuando además se quiere clonar la voz de una persona concreta, se incorpora una señal adicional:

texto + información del hablante → síntesis condicionada → audio con identidad vocal específica

Esta arquitectura modular ayuda a entender por qué la generación de voz es un problema complejo: no basta con pronunciar correctamente las palabras, sino que también hay que modelar el tiempo, la prosodia, el timbre y, en muchos casos, la identidad del hablante. Precisamente, uno de los principales avances del deep learning en este campo ha sido lograr que todos estos factores se aprendan de forma conjunta y con una calidad cada vez más cercana a la voz humana.

## Coqui TTS

Coqui TTS es una biblioteca de código abierto orientada a la síntesis de voz mediante aprendizaje profundo. Su principal interés en el contexto de este trabajo es que proporciona una base unificada para utilizar distintos modelos de generación de voz, realizar inferencia con modelos preentrenados y, en entornos más avanzados, entrenar o ajustar modelos propios.

Una de las fortalezas de Coqui TTS es que no se limita a ofrecer un único sistema cerrado, sino que reúne varias piezas del ecosistema de síntesis de voz. Dentro de la biblioteca pueden encontrarse modelos de predicción acústica, vocoders neuronales, mecanismos de codificación del hablante y utilidades relacionadas con el análisis y procesamiento de conjuntos de datos. Esto la convierte en una herramienta útil tanto para fines docentes como para experimentación aplicada.

Desde un punto de vista práctico, Coqui TTS facilita trabajar con distintas configuraciones de generación de voz, por ejemplo:

* modelos de un solo hablante;
* modelos multihablante;
* modelos multilingües;
* sistemas de clonación de voz;
* y tareas próximas, como la conversión de voz.

Este carácter modular hace que la librería sea especialmente interesante para explicar la generación de voz en un entorno académico. En lugar de tratar la síntesis de voz como una “caja negra”, Coqui TTS permite mostrar que detrás del audio final existen distintos componentes especializados y diferentes enfoques de modelado.

Otra ventaja importante es su orientación a la reproducibilidad y a la facilidad de uso. La librería permite cargar modelos preentrenados desde Python con pocas líneas de código, generar audios a partir de texto y experimentar con parámetros como el idioma o la voz de referencia. Esto reduce la complejidad de la implementación y permite centrar el trabajo en el análisis conceptual y experimental, sin necesidad de entrenar un modelo desde cero.

Para este trabajo, Coqui TTS se utilizará como entorno principal de experimentación porque permite ilustrar de forma clara la evolución de la síntesis de voz moderna: desde modelos TTS convencionales hasta sistemas más avanzados capaces de sintetizar en varios idiomas y aproximar el timbre de un hablante concreto a partir de un ejemplo de voz.

## XTTS y generación multilingüe

Dentro del ecosistema de Coqui TTS, uno de los modelos más interesantes para estudiar la generación de voz es XTTS, ya que combina varias de las capacidades más relevantes de los sistemas actuales: síntesis de voz a partir de texto, funcionamiento multilingüe y clonación de voz condicionada por una muestra de referencia.

La idea central de este tipo de modelos es que la voz generada no depende únicamente del texto de entrada. Además del contenido lingüístico, el sistema puede recibir información sobre qué idioma debe utilizar y cómo debe sonar la voz resultante. Para ello, se emplea un audio de referencia del hablante objetivo, a partir del cual el modelo extrae información relacionada con el timbre, el estilo y otras características vocales. De este modo, el sistema puede generar una frase nueva, incluso en otro idioma, manteniendo de forma aproximada la identidad vocal del hablante de referencia.

Esta capacidad representa un avance significativo respecto a sistemas TTS más clásicos. En un modelo convencional, la voz suele estar fijada por el propio entrenamiento y el usuario únicamente controla el texto. En cambio, en un sistema como XTTS el proceso de síntesis está condicionado por varios factores a la vez:

* el contenido textual,
* el idioma de salida,
* y una representación del hablante obtenida a partir de un ejemplo de audio.

Gracias a este diseño, XTTS resulta especialmente adecuado para estudiar dos problemas de gran interés actual. El primero es la síntesis multilingüe, donde un mismo sistema puede generar voz en distintos idiomas sin necesidad de entrenar un modelo independiente para cada uno. El segundo es la clonación de voz de pocos ejemplos, donde se intenta reproducir la identidad vocal de una persona a partir de una muestra relativamente breve.

Desde el punto de vista académico, XTTS permite ilustrar muy bien cómo el aprendizaje profundo ha ampliado el alcance del TTS. Ya no se trata solo de producir voz inteligible, sino de dotar al modelo de mecanismos de control sobre atributos que antes eran difíciles de modificar, como la identidad del hablante o la transferencia entre idiomas. Esto convierte a XTTS en un caso de estudio especialmente representativo de la generación de voz moderna.

Sin embargo, estas capacidades también introducen nuevos retos. La calidad de la clonación puede depender de la limpieza del audio de referencia, de la duración de la muestra o de la similitud entre idiomas. Además, aunque el modelo pueda producir resultados muy convincentes, esto no implica una reproducción perfecta de la voz original. En la práctica, la salida suele ser una aproximación que conserva algunos rasgos característicos del hablante, pero que puede mostrar variaciones en prosodia, pronunciación o estabilidad.

En este trabajo, XTTS se utilizará como ejemplo principal para analizar cómo un sistema de deep learning puede generar voz natural, multilingüe y condicionada por una identidad vocal, permitiendo estudiar tanto sus posibilidades técnicas como sus limitaciones reales.

## Implementación

In [4]:
!python --version
!sudo apt-get update
!sudo apt-get install -y python3.11 python3.11-distutils
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!python --version

Python 3.11.13
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Get:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]

In [10]:
# Desinstalar numpy y TTS para eliminar cualquier instalación previa incompatible.
# Usamos `python3.11 -m pip` para asegurarnos de que el pip correcto se está utilizando.
!python3.11 -m pip uninstall -y numpy
!python3.11 -m pip uninstall -y TTS

# Reinstalar numpy (una versión compatible) y luego TTS.
# A veces, especificar una versión de numpy ayuda a evitar problemas de compatibilidad.
!python3.11 -m pip install numpy==1.26.4
!python3.11 -m pip install TTS==0.22.0

print("Numpy y TTS reinstalados. Por favor, vuelve a ejecutar la celda donde apareció el error (la que importa `TTS.api.TTS`).")

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: TTS 0.22.0
Uninstalling TTS-0.22.0:
  Successfully uninstalled TTS-0.22.0
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
cudf-cu12 25.2.1 requires pandas<2.2.4dev0,>=2.0, but you have pandas 1.5.3 which is incompatible.
dask-cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 1.5.3 which is incompatible.
dask-expr 1.1.21 requires pandas>=2, but you have pandas 1.5.3 which is incompatible.
mizani 0.13.5 requires pandas>=2.

In [1]:
!pip -q install TTS==0.22.0

In [2]:
import os
from pathlib import Path

from IPython.display import Audio, display
from TTS.api import TTS

In [3]:
BASE_DIR = Path("trabajo_tts")
REF_DIR = BASE_DIR / "referencias"
OUT_DIR = BASE_DIR / "salidas"

REF_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de referencias:", REF_DIR.resolve())
print("Carpeta de salidas:", OUT_DIR.resolve())

Carpeta de referencias: /content/trabajo_tts/referencias
Carpeta de salidas: /content/trabajo_tts/salidas


In [4]:
# Esta celda puede tardar un poco.
# Sirve para comprobar qué modelos detecta la librería en tu entorno.

tts_manager = TTS()
print("Objeto TTS inicializado correctamente.")

Objeto TTS inicializado correctamente.


In [13]:
!pip uninstall -y torch torchaudio torchvision
!pip install -q torch==2.5.1 torchaudio==2.5.1 torchvision==0.20.1

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.5/906.5 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 139.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 8.1 MB/s eta 0:00:00


In [14]:
import torch
print(torch.__version__)

2.6.0+cu124


In [15]:
model_name = "tts_models/multilingual/multi-dataset/xtts_v2"
tts = TTS(model_name)
print("Modelo cargado:", model_name)

 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL TTS.tts.configs.xtts_config.XttsConfig was not an allowed global by default. Please use `torch.serialization.add_safe_globals([XttsConfig])` or the `torch.serialization.safe_globals([XttsConfig])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by 

In [8]:
import os
from pathlib import Path
import urllib.request

REF_DIR = Path("trabajo_tts/referencias")
REF_DIR.mkdir(parents=True, exist_ok=True)

urls = {
    "voz_referencia_1.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p225.wav",
    "voz_referencia_2.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p228.wav",
    "voz_referencia_3.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p229.wav",
    "voz_referencia_4.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p230.wav",
    "voz_referencia_5.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p231.wav",
}

for nombre, url in urls.items():
    destino = REF_DIR / nombre
    urllib.request.urlretrieve(url, destino)
    print("Descargado:", destino)

print("\nArchivos disponibles:")
for f in sorted(REF_DIR.iterdir()):
    print("-", f.name)

Descargado: trabajo_tts/referencias/voz_referencia_1.wav
Descargado: trabajo_tts/referencias/voz_referencia_2.wav
Descargado: trabajo_tts/referencias/voz_referencia_3.wav
Descargado: trabajo_tts/referencias/voz_referencia_4.wav
Descargado: trabajo_tts/referencias/voz_referencia_5.wav

Archivos disponibles:
- voz_referencia_1.wav
- voz_referencia_2.wav
- voz_referencia_3.wav
- voz_referencia_4.wav
- voz_referencia_5.wav


In [11]:
speaker_wav = REF_DIR / "voz_referencia_1.wav"

print("Ruta esperada del archivo de referencia:")
print(speaker_wav.resolve())

Ruta esperada del archivo de referencia:
/content/trabajo_tts/referencias/voz_referencia_1.wav


In [12]:
texto_es = "Hola, esta es una prueba de generación de voz con aprendizaje profundo."
salida_es = OUT_DIR / "ejemplo_es.wav"

tts.tts_to_file(
    text=texto_es,
    file_path=str(salida_es),
    speaker_wav=str(speaker_wav),
    language="es"
)

print("Archivo generado en:", salida_es)
display(Audio(str(salida_es)))

NameError: name 'tts' is not defined

In [ ]:
texto_en = "Hello, this is a multilingual voice generation test using a cloned reference voice."
salida_en = OUT_DIR / "ejemplo_en.wav"

tts.tts_to_file(
    text=texto_en,
    file_path=str(salida_en),
    speaker_wav=str(speaker_wav),
    language="en"
)

print("Archivo generado en:", salida_en)
display(Audio(str(salida_en)))

## Experimentos

In [ ]:
def generar_audio(texto, idioma, nombre_salida, speaker_path):
    ruta_salida = OUT_DIR / nombre_salida
    tts.tts_to_file(
        text=texto,
        file_path=str(ruta_salida),
        speaker_wav=str(speaker_path),
        language=idioma
    )
    return ruta_salida

In [ ]:
# Mismo hablante, distintos idiomas

textos = {
    "es": "Este experimento evalúa la capacidad del modelo para mantener la identidad vocal en distintos idiomas.",
    "en": "This experiment evaluates whether the model preserves speaker identity across different languages.",
    "fr": "Cette expérience évalue si le modèle conserve l'identité vocale dans plusieurs langues."
}

rutas = {}
for idioma, texto in textos.items():
    nombre = f"exp1_{idioma}.wav"
    rutas[idioma] = generar_audio(texto, idioma, nombre, speaker_wav)

for idioma, ruta in rutas.items():
    print(f"{idioma}: {ruta}")
    display(Audio(str(ruta)))

In [ ]:
# Texto corto vs largo

texto_corto = "Hoy hace buen día."
texto_largo = (
    "Hoy hace buen día y vamos a aprovechar esta prueba para observar cómo cambia la entonación "
    "del modelo cuando la secuencia de entrada es más larga y contiene varias pausas naturales, "
    "comas y una estructura sintáctica algo más compleja."
)

ruta_corta = generar_audio(texto_corto, "es", "exp2_corto.wav", speaker_wav)
ruta_larga = generar_audio(texto_largo, "es", "exp2_largo.wav", speaker_wav)

print("Texto corto")
display(Audio(str(ruta_corta)))

print("Texto largo")
display(Audio(str(ruta_larga)))

In [ ]:
# Nombres propios y pronunciación

texto_nombres = (
    "María visitó OpenAI en San Francisco y después presentó su proyecto en la Universidad de Zaragoza."
)

ruta_nombres = generar_audio(texto_nombres, "es", "exp3_nombres.wav", speaker_wav)

print("Prueba de pronunciación de nombres propios:")
display(Audio(str(ruta_nombres)))

In [ ]:
# Distintas voces de referencias

speaker_wav_1 = REF_DIR / "voz_referencia.wav"
speaker_wav_2 = REF_DIR / "voz_referencia_2.wav"   # súbelo si quieres comparar

texto_comp = "Esta frase se utilizará para comparar dos voces de referencia diferentes con el mismo texto."

ruta_ref1 = generar_audio(texto_comp, "es", "exp4_ref1.wav", speaker_wav_1)

print("Resultado con la primera referencia:")
display(Audio(str(ruta_ref1)))

if speaker_wav_2.exists():
    ruta_ref2 = generar_audio(texto_comp, "es", "exp4_ref2.wav", speaker_wav_2)
    print("Resultado con la segunda referencia:")
    display(Audio(str(ruta_ref2)))
else:
    print("No se encontró 'voz_referencia_2.wav'. Sube un segundo audio si quieres ejecutar esta comparación.")

## Hugginface

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import create_repo

repo_id = "adcelis/voice-generation-demo"

create_repo(
    repo_id=repo_id,
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True
)

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="mi_space",
    repo_id=repo_id,
    repo_type="space"
)

## Conclusiones